# Collecting FIPI responses and hidden state vectors

Before selecting data, I let the base Qwen2.5-7B model respond to both the safe and injection versions of each candidate prompt, and collect the hidden state vectors that the classifiers use to score those prompts.

The model completes the original FIPI task, while the classifiers read the hidden state vectors at the last prompt token before response generation. This notebook examines the completed collection results.

- [Experiment settings](#Experiment-settings)
- [How the data are split](#How-the-data-are-split)
- [The model responses and inspected layers](#The-model-responses-and-inspected-layers)
- [Which pairs are eligible for selection?](#Which-pairs-are-eligible-for-selection?)


## Experiment settings

I use 1,500 FIPI pairs for probe training, another 500 for choosing the classifier layers, and 10,000 candidate pairs for data selection, while keeping FIPI's original testing split separate.

The dictionary variable `config` contains the Qwen2.5-7B settings, and the table displays the candidate count, selection size and token limits.


In [ ]:
# inspect the responses and hidden state positions used for FIPI selection

from pathlib import Path
import sys
import json
import pandas as pd

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "scripts/curation/configs/config_qwen7b.json").exists())
sys.path.insert(0, str(root))

# use the same Qwen2.5-7B configuration across all selection methods
from scripts.curation.selection.extract_results import configuration, selection, validation
config, output = configuration()
pd.set_option("display.max_colwidth", None)

display(pd.Series({key: config[key] for key in ["model", "candidate_pairs", "selected_pairs", "max_input_tokens", "max_new_tokens"]}, name="setting"))

## How the data are split

Several FIPI pairs can share the same original task and content, so I keep related source prompts together when dividing the data into probe training, validation and candidate groups.

The dictionary variable `manifest` records the counts for each group, along with the training records included or excluded during preparation.


In [ ]:
# keep variants of the same original prompt together when comparing dataset splits
manifest = json.loads((output / "manifest.json").read_text())
display(pd.Series(manifest["split_pairs"], name="pairs").to_frame())
display(pd.Series(manifest["training_row_use"], name="training rows").to_frame())

## The model responses and inspected layers

The dictionary variable `recording` shows which token position and layers were inspected, along with the model precision and generation settings.

For response generation, I use greedy decoding (choosing the highest-scoring next token) with a limit of 2,048 response tokens, and the table variable `responses` contains the resulting safe and injection responses.


In [ ]:
# the token position tells which hidden state vectors the selectors received
recording = json.loads((output / "recording.json").read_text())
display(pd.Series({key: recording[key] for key in ["token_position", "layers", "dtype", "generation"]}, name="recording").to_frame())

# collect both generated responses for each candidate pair
response_rows = []
for path in sorted((output / "responses/candidates").glob("*.jsonl")):
    with path.open() as stream:
        response_rows.extend({key: value for key, value in json.loads(line).items() if key != "response_ids"} for line in stream if line.strip())
responses = pd.DataFrame(response_rows)
display(responses.groupby("condition").size().rename("recorded responses").to_frame())

## Which pairs are eligible for selection?

I keep a candidate pair eligible only when both responses finish within the generation limit, leaving 9,987 eligible pairs out of the original 10,000 in this run.

The table variable `candidates` combines this completion check with the answer check, where the source field `witness` contains the answer requested by the injection. The column `witness_redirection` is true if that answer appears in the injection response and is absent from the safe response, and the final table displays four generated responses for inspection.


In [ ]:
with (output / "splits/candidates.jsonl").open() as stream:
    candidates = pd.DataFrame(json.loads(line) for line in stream if line.strip())

# keep a pair eligible only when both generated responses reached an end token
complete = responses.groupby("pair_id").finished.all()
# put both responses on one row so the same requested answer can be checked in each
answer_matches = responses.pivot(index="pair_id", columns="condition", values="witness_present")
candidates["eligible"] = candidates.pair_id.map(complete)
candidates["witness_redirection"] = candidates.pair_id.map(answer_matches.injection & ~answer_matches.safe)
display(responses.groupby("condition").agg(prompts=("pair_id", "size"), finished=("finished", "sum"), witness_matches=("witness_present", "sum")))
display(candidates[["eligible", "witness_redirection"]].sum().rename("pairs").to_frame())
display(responses.head(4)[["pair_id", "condition", "response", "finished"]])